In [1]:
import webcam
import sys
import time
import datasetMng
import json
from mqtt_wrapper import MQTTWrapper

# Libraries
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, Conv2d, AvgPool2d, BatchNorm2d, Dropout2d, LeakyReLU, Linear, Module
from torch.utils.data import DataLoader

from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder

C:\Users\lucrezia.rettori\AppData\Local\anaconda3\envs\lucrezia-env\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
datasetMng.dataTransformer()

In [4]:
# Build Model
class DistractionCNN(Module):
    def __init__(self):
        super(DistractionCNN, self).__init__()
        self.conv = Conv2d(in_channels=3, out_channels=8, kernel_size=(3,3), stride=1, padding=1)
        self.pool = AvgPool2d(kernel_size=(3,3), stride=1)
        self.relu = LeakyReLU()
        self.bn = BatchNorm2d(num_features=8)
        self.drop = Dropout2d(p=0.3)
        self.fc = Linear(in_features=8*126*126, out_features=5)
        
    def forward(self, X):
        output = self.conv(X)
        output = self.pool(output)
        output = self.relu(output)
        output = self.bn(output)
        output = self.drop(output)
#         print(output.shape)
        output = output.view(-1, 8*126*126)
        output = self.fc(output)
        return output

In [9]:
# Define Model and Migrate to GPU
model = DistractionCNN().to(device)
# Optimiser
optimiser = Adam(model.parameters(), lr=1e-4, weight_decay=0.0001)

# Loss Function
loss_fn = CrossEntropyLoss()

In [11]:
for epoch in range(25):
    start = time()
    train_acc = 0
    test_acc = 0

    # Train
    model.train()
    with tqdm(datasetMng.train_loader, unit="batch") as tepoch:
        for xtrain, ytrain in tepoch:
            optimiser.zero_grad()
            xtrain = xtrain.to(device)
            train_prob = model(xtrain)
            train_prob = train_prob.cpu()
            train_loss = loss_fn(train_prob, ytrain)
            train_loss.backward()
            optimiser.step()
            # END TRAIN
            train_pred = torch.max(train_prob, 1).indices
            train_acc += int(torch.sum(train_pred == ytrain))
            
        train_epoch_accuracy = train_acc / datasetMng.len_train

    # Evaluate
    model.eval()
    with torch.no_grad():
        for xtest, ytest in datasetMng.test_loader:
            xtest = xtest.to(device)
            test_prob = model(xtest)
            test_prob = test_prob.cpu()
            test_loss = loss_fn(test_prob, ytest)
            test_pred = torch.max(test_prob, 1).indices
            test_acc += int(torch.sum(test_pred == ytest))
        
        test_epoch_accuracy = test_acc / datasetMng.len_test
    
    end = time()
    
    diff = end - start
    
    print(f"Epoch: {epoch+1}, Time: {diff}\nTr_loss: {train_loss}, Test_loss: {test_loss}\n,Tr_acc:{train_epoch_accuracy}, Test_acc: {test_epoch_accuracy}")

100%|██████████| 505/505 [01:43<00:00,  4.90batch/s]


Epoch: 1, Time: 115.0003399848938
Tr_loss: 1.2405296564102173, Test_loss: 1.645327091217041
,Tr_acc:0.5563696369636963, Test_acc: 0.7059645852749301


100%|██████████| 505/505 [01:35<00:00,  5.28batch/s]


Epoch: 2, Time: 106.289621591568
Tr_loss: 1.307536005973816, Test_loss: 1.0956170558929443
,Tr_acc:0.6985478547854785, Test_acc: 0.7614165890027959


100%|██████████| 505/505 [01:35<00:00,  5.26batch/s]


...

Epoch: 23, Time: 107.07897019386292
Tr_loss: 0.3257061541080475, Test_loss: 0.055246803909540176
,Tr_acc:0.885016501650165, Test_acc: 0.9082013047530288


100%|██████████| 505/505 [01:35<00:00,  5.27batch/s]


Epoch: 24, Time: 106.54144811630249
Tr_loss: 0.3037467300891876, Test_loss: 0.2107652723789215
,Tr_acc:0.8911551155115511, Test_acc: 0.918452935694315


100%|██████████| 505/505 [01:36<00:00,  5.26batch/s]


Epoch: 25, Time: 106.42892146110535
Tr_loss: 1.0381388664245605, Test_loss: 0.04576284438371658
,Tr_acc:0.8906270627062707, Test_acc: 0.9114631873252563


In [ ]:
def loadUserConfig():
    with open(sys.path[0] + "/userConfig.json") as json_file:
        return json.load(json_file)

client = MQTTWrapper.fromJsonConfig(sys.path[0] + "/mqttConfig.json")
user = loadUserConfig()
webcam.setup()

while True:
    try:
        distractionsMap = webcam.getDistractionType()
        dataToTransmit = { 
            user["user"]: {
                "other_activities": distractionsMap["Other activities"],
                "safe_driving": distractionsMap["Safe driving"],
                "talking_phone": distractionsMap["Talking on the phone"],
                "texting_phone": distractionsMap["Texting on the phone"],
                "turning": distractionsMap["Turning"]
            } 
        }
        client.publish(json.dumps(dataToTransmit))
        print(dataToTransmit)
        time.sleep(1)
    except webcam.NoFaceDetectedException:
        print("No face found!")